# 06 — Analyse de la segmentation et recommandations

**Projet** : Segmentation client d'un e-commerçant (RFM étendu) avec scikit-learn
**Principe** : une silhouette globale ne dit **jamais** si la segmentation sert le métier. Un
clustering peut être géométriquement propre et produire des groupes inexploitables (micro-groupes,
profils illisibles, clients à la frontière). Ce notebook descend au niveau du **segment** puis du
**client** : qui est dans chaque groupe, qu'est-ce qui le définit, quels clients sont mal affectés,
et que fait-on lundi matin ?

Le split de **test** n'est utilisé qu'ici — une seule fois — pour rester une estimation honnête.

## Objectifs pédagogiques

1. Produire une évaluation complète : critères internes, tailles de groupes, confiance des affectations.
1. Profiler chaque segment **en unités brutes** (euros, jours, commandes) et identifier ses drivers en écarts-types.
1. Vérifier la **validité externe** : les groupes séparent-ils un comportement observé après coup (churn à 90 jours) ?
1. Repérer les affectations fragiles et les groupes dégénérés, puis formuler des recommandations chiffrées.

**Objectifs transverses du dépôt**

- Construire un pipeline non supervisé sans fuite : les colonnes de diagnostic sont exclues des features par configuration.
- Choisir le nombre de groupes par triangulation (silhouette, Davies-Bouldin, coude d'inertie, taille minimale).
- Comprendre l'effet de l'échelle et des queues de distribution sur une distance euclidienne (log, winsorising, standardisation).

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402
from loguru import logger  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
logger.remove()
logger.add(sys.stderr, level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (1500 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 1500

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=3",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.18)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "feature_names": list(pipeline.feature_names_out),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

## 1. Évaluation sur le split de test

In [ ]:
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"],
    None,
    split="test",
    context=PREPARED["enriched"]["test"],
)

metrics_frame = pd.DataFrame(
    {"métrique": list(RESULT.metrics), "valeur": [RESULT.metrics[name] for name in RESULT.metrics]}
)
print(f"clients évalués          : {RESULT.n_samples}")
print(f"groupes produits         : {RESULT.n_clusters}")
print(f"plus petit groupe        : {RESULT.min_cluster_share:.1%}")
print(f"stabilité (ARI moyen)    : {RESULT.stability_ari:.3f}")
print(f"accord latent (ARI)      : {RESULT.ari_latent:.3f}")
print(f"écart de churn           : {RESULT.churn_spread:.1%}")
print(f"groupes dégénérés        : {RESULT.degenerate_clusters or 'aucun'}")
metrics_frame.round(4)

**Ce qu'il faut retenir**

- La métrique de décision est `silhouette` = **celle affichée ci-dessus** — c'est elle qui pilote le seuil de qualité (`metrics.min_primary`).
- Les critères internes (silhouette, Davies-Bouldin, Calinski-Harabasz) jugent la **géométrie** ; la taille des groupes et la stabilité jugent l'**exploitabilité** ; le churn observé juge l'**utilité métier**.
- Un seul de ces trois regards ne suffit jamais : c'est la raison d'être de ce notebook.

## 2. Profils des segments (unités brutes)

In [ ]:
profiles = RESULT.per_segment
preferred = {
    "cluster",
    "size",
    "share",
    "mean_distance",
    "mean_silhouette",
    "churn_rate",
    "dominant_latent",
    "latent_purity",
}
display_columns = [
    column for column in profiles.columns if column in preferred or str(column).startswith("mean_")
]
display(profiles[display_columns].round(3))

revenue_column = next(
    (column for column in profiles.columns if str(column).endswith("revenue_12m_eur")), None
)
if revenue_column:
    contribution = (profiles[revenue_column] * profiles["share"]).sort_values(ascending=False)
    total = float(contribution.sum())
    print("Contribution relative au chiffre d'affaires (à lire en relatif) :")
    for cluster, value in contribution.items():
        share = value / total if total else float("nan")
        print(f"  segment {cluster} : {share:.1%}")

**Ce qu'il faut retenir**

- Un profil se lit en **unités métier** : panier moyen en euros, récence en jours, part promotionnelle en pourcentage. Les écarts-types standardisés du modèle ne parlent ni au CRM ni au marketing.
- La colonne `dominant_latent` (diagnostic) montre quel profil injecté le groupe retrouve, et `latent_purity` la part de ce profil dans le groupe : une pureté de 60 % signifie que 4 clients sur 10 viennent d'ailleurs.
- La concentration de la valeur est l'argument de priorisation : si 8 % des clients portent 40 % du chiffre d'affaires, la rétention de ce groupe passe avant toute acquisition.

## 3. Ce qui définit chaque segment (écarts-types)

In [ ]:
centroids = RESULT.feature_importance
top_drivers = (
    centroids[centroids["rank"] <= 4]
    .pivot_table(index="cluster", columns="feature", values="centroid_z")
    .round(2)
)
display(top_drivers)

pivot = centroids.pivot_table(index="feature", columns="cluster", values="centroid_z")
strength = centroids.groupby("feature")["centroid_z"].apply(
    lambda series: float(series.abs().max())
)
keep = strength.sort_values(ascending=False).head(12).index.tolist()
pivot = pivot.loc[[name for name in keep if name in pivot.index]]

fig, axis = plt.subplots(figsize=(0.9 * len(pivot.columns) + 6.4, 0.42 * len(pivot) + 2.0))
image = axis.imshow(pivot.to_numpy(), cmap="RdBu_r", vmin=-2.5, vmax=2.5, aspect="auto")
axis.set_xticks(
    range(len(pivot.columns)), [f"segment {name}" for name in pivot.columns], fontsize=8
)
axis.set_yticks(range(len(pivot.index)), pivot.index, fontsize=8)
for row in range(pivot.shape[0]):
    for column in range(pivot.shape[1]):
        value = pivot.iloc[row, column]
        axis.text(
            column,
            row,
            f"{value:+.1f}",
            ha="center",
            va="center",
            fontsize=7,
            color="black" if abs(value) < 1.4 else "white",
        )
fig.colorbar(image, ax=axis, fraction=0.035, pad=0.03, label="écart-type vs client moyen")
axis.set_title("Profil des segments : ce qui distingue chaque groupe")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- Les coordonnées du centroïde sont des **tailles d'effet** : `+1,8` sur la part promotionnelle signifie « 1,8 écart-type au-dessus du client moyen ». C'est ce qui rend un groupe nommable.
- Un groupe dont aucun driver ne sort de ±0,5 écart-type est un groupe « moyen » : il n'a pas de personnalité et doit probablement être fusionné.
- Vigilance sur les variables **dérivées** (ratios, panier moyen, palier de fidélité) : si elles dominent un centroïde, la segmentation redécouvre une identité arithmétique ou une règle métier, pas un comportement.

## 4. Validité externe et stabilité

In [ ]:
external = dict(RESULT.extras.get("external_validity", {}))
stability = dict(RESULT.extras.get("stability", {}))

print("Accord avec la structure latente (diagnostic pédagogique) :")
print(f"  ARI        : {float(external.get('ari_latent', float('nan'))):.3f}")
print(f"  NMI        : {float(external.get('nmi_latent', float('nan'))):.3f}")
print(f"  V-measure  : {float(external.get('v_measure_latent', float('nan'))):.3f}")

churn = external.get("churn_by_cluster", {})
if churn:
    churn_frame = pd.DataFrame(
        {"segment": list(churn), "churn_90j": [float(value) * 100 for value in churn.values()]}
    ).sort_values("churn_90j", ascending=False)
    display(churn_frame.round(1))
    fig, axis = plt.subplots(figsize=(7.6, 3.8))
    axis.bar(churn_frame["segment"].astype(str), churn_frame["churn_90j"], color="#ae2012")
    axis.set_ylabel("churn à 90 jours (%)")
    axis.set_xlabel("segment")
    axis.set_title("Validité externe : la segmentation sépare-t-elle un risque réel ?")
    fig.tight_layout()
    plt.show()

print("Stabilité des affectations (ARI contre l'affectation de référence) :")
for seed, value in dict(stability.get("per_seed", {})).items():
    print(f"  graine {seed} : {float(value):.3f}")
print(f"  moyenne : {float(stability.get('mean_ari', float('nan'))):.3f}")

**Ce qu'il faut retenir**

- L'ARI latent n'est **pas** un objectif de production : personne ne connaît la vraie segmentation. C'est un outil de mise au point qui dit si la méthode retrouve une structure connue.
- Le churn observé à 90 jours est la vraie validation : une segmentation dont les groupes diffèrent de 20 points de churn pilote directement une campagne de rétention.
- Une stabilité faible (ARI < 0.85) interdit la mise en production, quelle que soit la silhouette : les clients changeraient de groupe à chaque ré-entraînement.

## 5. Affectations fragiles et carte des clients

In [ ]:
rows = RESULT.predictions
confidence = rows["confidence"].value_counts(normalize=True)
print("Confiance des affectations :")
display((confidence * 100).round(1).to_frame("%"))

borderline = RESULT.errors
display_columns = [
    column
    for column in ("cluster", "silhouette", "distance_to_centroid", "confidence")
    if column in borderline.columns
]
display(borderline[display_columns].head(12).round(3))

negative_share = float((pd.to_numeric(rows["silhouette"], errors="coerce") < 0).mean())
print(f"clients plus proches d'un autre groupe (silhouette < 0) : {negative_share:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.2))
values = pd.to_numeric(rows["silhouette"], errors="coerce").dropna()
axes[0].hist(values, bins=36, color="#0a9396", edgecolor="white")
axes[0].axvline(
    float(values.mean()), color="#ae2012", linestyle="--", label=f"moyenne {values.mean():.3f}"
)
axes[0].axvline(0.0, color="#495057", linestyle=":")
axes[0].legend(fontsize=8)
axes[0].set_title("Silhouette individuelle")
axes[0].set_xlabel("silhouette")

projection = dict(RESULT.curves.get("pca_map", {}))
if projection.get("x"):
    scatter_frame = pd.DataFrame(
        {
            "x": projection["x"],
            "y": projection["y"],
            "segment": [str(value) for value in projection["cluster"]],
        }
    )
    for segment, group in scatter_frame.groupby("segment", observed=True):
        axes[1].scatter(
            group["x"], group["y"], s=12, alpha=0.55, label=f"segment {segment}", edgecolors="none"
        )
    axes[1].legend(fontsize=7, markerscale=1.8, title="segment", title_fontsize=7)
axes[1].set_title("Carte ACP des clients évalués")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
fig.tight_layout()
plt.show()

**Ce qu'il faut retenir**

- La part de clients à silhouette négative est plus informative que la moyenne : ce sont les clients **à la frontière**, ceux pour lesquels une campagne ciblée est un pari.
- La carte ACP rend le chevauchement visible : des groupes qui s'interpénètrent sur le plan ne sont pas nécessairement mal séparés en dimension supérieure, mais ils signalent un continuum comportemental.
- En production, ces clients doivent déclencher une règle métier (campagne générique, revue humaine) plutôt qu'un message personnalisé : c'est le rôle du niveau de confiance publié par le prédicteur.

## 6. Diagnostics : ce qui affaiblit cette segmentation

In [ ]:
from src.evaluation.reports import ReportBuilder

BUILDER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
diagnostics = BUILDER.diagnostics(RESULT)

if not diagnostics:
    print("Aucun signal de faiblesse marqué sur ce split.")
for title, detail in diagnostics:
    print(f"\n### {title}\n{detail}")

print("\n--- seuils du cas d'usage ---")
for name, value in sorted(BUILDER.thresholds.items()):
    print(f"  {name}: {value}")

**Ce qu'il faut retenir**

- Les diagnostics sont **calculés**, pas récités : chaque faiblesse observée (groupe dégénéré, instabilité, faible validité externe, frontières floues, variable circulaire) déclenche sa propre hypothèse.
- Les seuils viennent de la configuration (`metrics.thresholds`) : un critère de qualité qui vit dans le code n'est pas auditable.
- Un diagnostic sans chiffre associé n'est pas une hypothèse, c'est une opinion.

## 7. Recommandations

In [ ]:
recommendations = BUILDER.recommendations(RESULT)
for index, item in enumerate(recommendations, start=1):
    print(f"{index}. {item}")

### déclenchées par les chiffres observés

Les recommandations ci-dessus sont produites par `ReportBuilder.recommendations()` : les premières
répondent aux diagnostics mesurés sur ce split, les suivantes sont les bonnes pratiques du cas
d'usage.

### documentées pour ce cas d'usage

1. Standardiser (et log-transformer les variables asymétriques) **avant** tout clustering : sans cela, la distance euclidienne est dominée par le chiffre d'affaires.
2. Choisir k par triangulation : coude d'inertie, maximum de silhouette, Davies-Bouldin, taille minimale des groupes et lisibilité métier — jamais par un seul indicateur.
3. Comparer KMeans (sphérique, rapide), MiniBatchKMeans (grands volumes) et GaussianMixture (groupes ellipsoïdaux, probabilités d'appartenance) sur les mêmes données et les mêmes métriques.
4. Mesurer la **stabilité** : ré-entraîner avec plusieurs graines et comparer les affectations (ARI). Une segmentation instable ne peut pas piloter des campagnes.
5. Valider en externe : taux de churn, panier moyen et part de CA par groupe doivent être nettement différenciés, sinon la segmentation n'apporte rien au CRM.
6. Profiler chaque groupe en langage métier (effectif, part du CA, comportement dominant) et lui associer une action : c'est la condition d'adoption.
7. Exclure les variables circulaires (`loyalty_tier`) ou les traiter comme descriptives et non comme features.
8. Publier une affectation **assortie d'une confiance** : les clients à la frontière de deux groupes doivent déclencher une règle métier (ou rester dans un groupe par défaut), pas une campagne ciblée.
9. Surveiller la dérive : la distribution des distances et la taille des groupes changent avec la saisonnalité ; une revue mensuelle est nécessaire.

### plan d'action proposé

| Priorité | Action | Effet attendu | Comment vérifier |
| --- | --- | --- | --- |
| 1 | Nommer chaque segment et lui associer une action CRM (message, canal, remise) | segmentation adoptée par le métier | `segmentation.labels` dans `conf/config.yaml` |
| 2 | Router les clients à confiance faible vers une campagne générique | baisse des messages inadaptés | `artifacts/reports/segment_assignments.csv` |
| 3 | Stabiliser l'entraînement (graine fixée, `n_init` augmenté) avant mise en production | ARI inter-graines ≥ 0.85 | notebook 04, §4 |
| 4 | Retirer les variables circulaires (palier de fidélité) de l'espace de description | segmentation comportementale, pas arithmétique | notebook 06, §3 (drivers) |
| 5 | Instrumenter la dérive (taille des groupes, distribution des distances) | alerte précoce | `mlops/model-monitoring` |
| 6 | Rejouer ce notebook à chaque nouvelle version de données | non-régression | `make evaluate` + CI |

## 8. Limites assumées

- Les données sont **synthétiques** : les niveaux de qualité illustrent une méthode, pas une base clients réelle.
- `latent_segment` et `churned_next_90d` sont des **métadonnées** : exclues des features, elles ne servent qu'au diagnostic. En production, aucune des deux n'existe au moment de segmenter.
- Une seule passe d'évaluation sur un split unique : la dérive temporelle de la base clients n'est pas mesurée ici.
- Le générateur mélange volontairement une partie des clients entre deux profils : une silhouette très élevée signerait une structure artificielle, pas une bonne méthode.
- L'analyse porte sur 6000 clients générés, dont une fraction en test : les segments rares restent peu observés.

**Aller plus loin dans le dépôt** : comparaison multi-stacks (`data-science/regression/with-*`,
`data-science/classification/with-*`), mise en production et suivi (`mlops/`), pipelines de données
(`data-eng/`).

## 9. Génération du rapport et des figures

In [ ]:
artifacts = BUILDER.build(RESULT, model=MODEL)
for kind, path in sorted(artifacts.items()):
    relative = path.relative_to(PROJECT_ROOT) if path.is_absolute() else path
    print(f"{kind:<12} -> {relative}")

display(Markdown(f"### Rapport généré\n\n`{artifacts['report']}`"))
if "profile_heatmap" in artifacts:
    display(Image(filename=str(artifacts["profile_heatmap"]), width=760))

**Ce qu'il faut retenir**

- Le rapport Markdown, son équivalent JSON, les profils CSV et les figures sont des **artefacts versionnables** : ils rendent la segmentation auditable six mois plus tard.
- `make evaluate` rejoue exactement cette chaîne de façon non interactive — c'est le même code, pas une copie.